In [1]:
# importing libries
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import seaborn as sns
import re, nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from collections import Counter
from wordcloud import WordCloud
from nltk import PorterStemmer, WordNetLemmatizer
from nltk import sent_tokenize
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


#

In [2]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


/kaggle/input/datasets/niraliivaghani/flipkart-product-customer-reviews-dataset/Dataset-SA.csv


In [3]:
#Loading the dataset
senti_data = pd.read_csv("/kaggle/input/datasets/niraliivaghani/flipkart-product-customer-reviews-dataset/Dataset-SA.csv")

In [4]:
#viewing the data
senti_data.head()

,product_name,product_price,Rate,Review,Summary,Sentiment
0,Candes 12 L Room/Personal Air Cooler??????(Whi...,3999,5,super!,great cooler excellent air flow and for this p...,positive
1,Candes 12 L Room/Personal Air Cooler??????(Whi...,3999,5,awesome,best budget 2 fit cooler nice cooling,positive
2,Candes 12 L Room/Personal Air Cooler??????(Whi...,3999,3,fair,the quality is good but the power of air is de...,positive
3,Candes 12 L Room/Personal Air Cooler??????(Whi...,3999,1,useless product,very bad product its a only a fan,negative
4,Candes 12 L Room/Personal Air Cooler??????(Whi...,3999,3,fair,ok ok product,neutral


In [5]:
#checking the number of rows and column in the dataset
senti_data.shape

(205052, 6)

In [6]:
senti_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 205052 entries, 0 to 205051
Data columns (total 6 columns):
 #   Column         Non-Null Count   Dtype 
---  ------         --------------   ----- 
 0   product_name   205052 non-null  object
 1   product_price  205052 non-null  object
 2   Rate           205052 non-null  object
 3   Review         180388 non-null  object
 4   Summary        205041 non-null  object
 5   Sentiment      205052 non-null  object
dtypes: object(6)
memory usage: 9.4+ MB


In [7]:
# number of missing values
senti_data.isnull().sum()

product_name         0
product_price        0
Rate                 0
Review           24664
Summary             11
Sentiment            0
dtype: int64

In [8]:
# percentage of missing values
senti_data.isnull().mean()*100

product_name      0.000000
product_price     0.000000
Rate              0.000000
Review           12.028168
Summary           0.005364
Sentiment         0.000000
dtype: float64

In [9]:
#Checking the unique values of in each column
senti_data.nunique()

product_name       958
product_price      525
Rate                 8
Review            1324
Summary          92923
Sentiment            3
dtype: int64

In [10]:
# uncdrsta
senti_data['product_price'].unique()

array(['3999', '8999', '7999', '9999', '1199', '499', '1999', '1099',
       '997', '1499', '435', '1349', '30999', '13999', '9990', '14299',
       '5298', '7599', '11999', '329', '425', '249', '302', '59', '245',
       '79', '349', '449', '340', '299', '469', '26990', '23479', '29390',
       '29990', '20990', '44490', '25990', '38490', '42000', '44890',
       '31590', '41990', '52990', '33990', '18990', '50999', '45550',
       '1401', '359', '1453', '254', '205', '1256', '1547', '195', '575',
       '366', '209', '219', '549', '859', '210', '215', '1142', '235',
       '221', '1599', '2454', '6099', '2399', '599', '849', '699',
       'pigeon favourite electric kettle15 l silver black', '4449',
       '4098', '5599', '5499', '1448', '3569', '2879', '1799', '1329',
       '5390', '11500', '1220', '9050', '6505', '6495', '11595', '7649',
       '4399', '6029', '6299', '5919', '6390', '2695', '2949', '7909',
       '4499', '6525', '6589', '5039', '4219', '4319', '7499', '379',
     

After checking the unique values of product name and product price has a lot of details and inconsistencies  and the price column is meant to an integer or a float

### Cleaning Data

In [11]:
# CLEANING
senti_data['clean_name'] = (
    senti_data['product_name']
    .str.replace(r'\?+', '', regex=True)
    .str.replace(r'�', '', regex=True)
    .str.strip()
    .str.replace(r'\s+', ' ', regex=True)
    .str.title()
)

# BRAND
senti_data['brand'] = senti_data['clean_name'].str.split().str[0]

# ATTRIBUTES
senti_data['attributes'] = senti_data['clean_name'].str.extract(r'\((.*?)\)')

# MAIN PRODUCT
senti_data['product_main'] = (
    senti_data['clean_name']
    .str.replace(r'\(.*?\)', '', regex=True)
    .str.strip()
)

# REMOVE DUPLICATES
senti_data = senti_data.drop_duplicates(subset='product_main')

In [12]:
senti_data.head()

,product_name,product_price,Rate,Review,Summary,Sentiment,clean_name,brand,attributes,product_main
0,Candes 12 L Room/Personal Air Cooler??????(Whi...,3999,5,super!,great cooler excellent air flow and for this p...,positive,"Candes 12 L Room/Personal Air Cooler(White, Bl...",Candes,"White, Black, Elegant High Speed-Honey Comb Co...",Candes 12 L Room/Personal Air Cooler
10,Candes 60 L Room/Personal Air Cooler??????(Whi...,8999,5,great product,beautiful product good material and perfectly ...,positive,"Candes 60 L Room/Personal Air Cooler(White, Bl...",Candes,"White, Black, Creta",Candes 60 L Room/Personal Air Cooler
36,MAHARAJA WHITELINE 65 L Desert Air Cooler?????...,7999,5,wonderful,the best poduct that i bought from flifkart it...,positive,Maharaja Whiteline 65 L Desert Air Cooler(Whit...,Maharaja,"White, Grey, Rambo Grey / Ac-303",Maharaja Whiteline 65 L Desert Air Cooler
816,"Crompton 75 L Desert Air Cooler??????(White, T...",9999,5,simply awesome,its really worth every single penny it works l...,positive,"Crompton 75 L Desert Air Cooler(White, Teal, A...",Crompton,"White, Teal, Acgc-Dac751",Crompton 75 L Desert Air Cooler
1999,boAt Rockerz 510 Super Extra Bass Bluetooth He...,1199,5,great product,my one request plz to all buyersplz dont revie...,positive,Boat Rockerz 510 Super Extra Bass Bluetooth He...,Boat,"Molten Orange, On The Ear",Boat Rockerz 510 Super Extra Bass Bluetooth He...


In [13]:
# Creating a copy of the data
senti_data = senti_data.copy()


In [14]:
# extracting the capacity
senti_data['capacity_liters'] = (
    senti_data['clean_name']
    .str.extract(r'(\d+)\s?L', expand=False)
    .astype('float')
)

In [15]:
# Extract Wattage
senti_data['wattage'] = senti_data['clean_name'].str.extract(r'(\d+\s?W)')

In [16]:
# Extract Color
senti_data['color'] = senti_data['attributes'].str.split(',').str[0]

In [17]:
senti_data.head()

,product_name,product_price,Rate,Review,Summary,Sentiment,clean_name,brand,attributes,product_main,capacity_liters,wattage,color
0,Candes 12 L Room/Personal Air Cooler??????(Whi...,3999,5,super!,great cooler excellent air flow and for this p...,positive,"Candes 12 L Room/Personal Air Cooler(White, Bl...",Candes,"White, Black, Elegant High Speed-Honey Comb Co...",Candes 12 L Room/Personal Air Cooler,12.0,NaN,White
10,Candes 60 L Room/Personal Air Cooler??????(Whi...,8999,5,great product,beautiful product good material and perfectly ...,positive,"Candes 60 L Room/Personal Air Cooler(White, Bl...",Candes,"White, Black, Creta",Candes 60 L Room/Personal Air Cooler,60.0,NaN,White
36,MAHARAJA WHITELINE 65 L Desert Air Cooler?????...,7999,5,wonderful,the best poduct that i bought from flifkart it...,positive,Maharaja Whiteline 65 L Desert Air Cooler(Whit...,Maharaja,"White, Grey, Rambo Grey / Ac-303",Maharaja Whiteline 65 L Desert Air Cooler,65.0,NaN,White
816,"Crompton 75 L Desert Air Cooler??????(White, T...",9999,5,simply awesome,its really worth every single penny it works l...,positive,"Crompton 75 L Desert Air Cooler(White, Teal, A...",Crompton,"White, Teal, Acgc-Dac751",Crompton 75 L Desert Air Cooler,75.0,NaN,White
1999,boAt Rockerz 510 Super Extra Bass Bluetooth He...,1199,5,great product,my one request plz to all buyersplz dont revie...,positive,Boat Rockerz 510 Super Extra Bass Bluetooth He...,Boat,"Molten Orange, On The Ear",Boat Rockerz 510 Super Extra Bass Bluetooth He...,NaN,NaN,Molten Orange
